# Interpretable Rule Learning in PyTorch
### Course: Interpretable Machine Learning — IIT Kharagpur
**Group 11**

| Member | Roll | Stage |
|---|---|---|
| Aritra Bose | 25EC65R08 | Data & Preprocessing |
| Chunarkar Sujeeth Kumar | 24AI10021 | Sequential Covering |
| Mohammad Asad ur Rahman Nisar | 24AI10038 | Bayesian Rule List |
| Kotikala Yashwant | 24AI10034 | Evaluation & Integration |

---
**Topic:** Sequential Covering (RIPPER-style) and Bayesian Rule Lists, implemented from scratch in PyTorch.

**Dataset:** UCI Adult Census Income — predict whether annual income exceeds \$50K.

## ⚙️ Setup & Imports

In [ ]:
# Standard library
import math
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')  # headless — change to 'inline' in Jupyter
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score,
                             roc_curve, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB

print(f'torch  {torch.__version__}')
print(f'pandas {pd.__version__}')
print(f'numpy  {np.__version__}')

---
## Stage 1 — Data Loading & Baselines
*Author: Aritra Bose (25EC65R08)*

We use the **UCI Adult Census Income** dataset. Key cleaning steps:
- `'?'` entries are treated as missing and rows are dropped (~7.4%)
- Target label normalised (trailing `.` stripped from test-file variant)
- Class imbalance: 75% ≤50K, 25% >50K — accuracy alone is misleading

In [ ]:
COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

DATA_PATH = 'data/adult.data'   # adjust if needed

def load_adult(file_path=DATA_PATH, verbose=True):
    """Read UCI Adult file, treat '?' as missing, drop incomplete rows, encode target."""
    df = pd.read_csv(file_path, names=COLUMNS, sep=r',\s*', engine='python',
                     na_values=['?', ' ?'])
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()
    n_before = len(df)
    df = df.dropna().reset_index(drop=True)
    n_after = len(df)
    df['income'] = df['income'].astype(str).str.rstrip('.')
    y = (df['income'] == '>50K').astype(int).values
    X = df.drop(columns=['income'])
    if verbose:
        print(f'Rows before dropna : {n_before}')
        print(f'Rows after  dropna : {n_after}')
        print(f'Dropped            : {n_before - n_after} ({(n_before-n_after)/n_before:.1%})')
        pos = y.mean()
        print(f'Class balance      : <=50K {1-pos:.2%} | >50K {pos:.2%}')
    return X, y

X_raw, y = load_adult(DATA_PATH)

### 1.2 Baselines

In [ ]:
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_raw, y, test_size=0.20, random_state=42, stratify=y)

# Majority-class baseline
maj_pred = np.zeros_like(y_te)
print('--- Majority-class baseline ---')
print(f'  Accuracy  : {accuracy_score(y_te, maj_pred):.4f}')
print(f'  Precision : {precision_score(y_te, maj_pred, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_te, maj_pred, zero_division=0):.4f}')
print(f'  F1        : {f1_score(y_te, maj_pred, zero_division=0):.4f}')

# Naive Bayes baseline
def fit_codes(frame):
    return {col: {v: i for i, v in enumerate(pd.unique(frame[col]))} for col in frame.columns}

def apply_codes(frame, codes):
    out = np.zeros((len(frame), len(frame.columns)), dtype=int)
    for j, col in enumerate(frame.columns):
        out[:, j] = [codes[col].get(v, 0) for v in frame[col]]
    return out

codes = fit_codes(X_tr_raw)
nb = CategoricalNB().fit(apply_codes(X_tr_raw, codes), y_tr)
nb_prob_raw = nb.predict_proba(apply_codes(X_te_raw, codes))[:, 1]
nb_pred = (nb_prob_raw >= 0.5).astype(int)
print('\n--- Naive Bayes baseline ---')
print(f'  Accuracy  : {accuracy_score(y_te, nb_pred):.4f}')
print(f'  Precision : {precision_score(y_te, nb_pred, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_te, nb_pred, zero_division=0):.4f}')
print(f'  F1        : {f1_score(y_te, nb_pred, zero_division=0):.4f}')
print(f'  AUC       : {roc_auc_score(y_te, nb_prob_raw):.4f}')

---
## Stage 2 — Discretisation & Literal Encoding
*Author: Aritra Bose (25EC65R08)*

Rule learners need propositional literals like `[age_group = Young]`.
Four continuous columns are binned into named intervals.
Two redundant/unreadable columns (`fnlwgt`, `education-num`) are dropped.
`native-country` is collapsed to *United-States vs Other*.

The dataset is then encoded as a boolean PyTorch tensor **M** of shape `(literals, examples)`.

In [ ]:
AGE_BINS    = [0, 25, 45, 65, 120]
AGE_LABELS  = ['Young (<=25)', 'Middle-aged (26-45)', 'Senior-Adult (46-65)', 'Senior (>65)']
HOURS_BINS  = [0, 34, 40, 55, 120]
HOURS_LABELS= ['Part-time (<35)', 'Full-time (35-40)', 'Overtime (41-55)', 'Extreme (>55)']
GAIN_BINS   = [-1, 0, 7000, 1_000_000]
GAIN_LABELS = ['None', 'Moderate (<7k)', 'High (>7k)']
LOSS_BINS   = [-1, 0, 1800, 1_000_000]
LOSS_LABELS = ['None', 'Low-Loss (<=1.8k)', 'High-Loss (>1.8k)']

def discretise(df):
    df = df.copy()
    df = df.drop(columns=['fnlwgt', 'education-num'])
    df['age_group']          = pd.cut(df['age'].astype(float),
                                       bins=AGE_BINS, labels=AGE_LABELS, right=True).astype(str)
    df['hours_group']        = pd.cut(df['hours-per-week'].astype(float),
                                       bins=HOURS_BINS, labels=HOURS_LABELS, right=True).astype(str)
    df['capital_gain_group'] = pd.cut(df['capital-gain'].astype(float),
                                       bins=GAIN_BINS, labels=GAIN_LABELS, right=True).astype(str)
    df['capital_loss_group'] = pd.cut(df['capital-loss'].astype(float),
                                       bins=LOSS_BINS, labels=LOSS_LABELS, right=True).astype(str)
    df = df.drop(columns=['age', 'hours-per-week', 'capital-gain', 'capital-loss'])
    df['native_country_group'] = np.where(df['native-country'] == 'United-States',
                                           'United-States', 'Other')
    df = df.drop(columns=['native-country'])
    return df

X_disc = discretise(X_raw)
print(f'Features: {X_raw.shape[1]} -> {X_disc.shape[1]}')
print('Columns:', list(X_disc.columns))

In [ ]:
class LiteralEncoder:
    """Encode a categorical DataFrame as a boolean PyTorch tensor M.

    M[i, j] == True  <=>  literal i is satisfied by example j.
    A rule is the element-wise AND of its selected rows in M.
    """
    def fit(self, X_df):
        self.columns_    = list(X_df.columns)
        self.literals_   = [(c, v) for c in self.columns_ for v in pd.unique(X_df[c])]
        self.col_index_  = {c: i for i, c in enumerate(self.columns_)}
        self.literal_col_= torch.tensor(
            [self.col_index_[c] for c, _ in self.literals_], dtype=torch.long)
        return self

    def encode(self, X_df):
        rows = [X_df[c].values == v for c, v in self.literals_]
        return torch.from_numpy(np.stack(rows))   # bool (L, N)

    def rule_mask(self, M, literal_idx):
        if len(literal_idx) == 0:
            return torch.ones(M.shape[1], dtype=torch.bool)
        return M[list(literal_idx)].all(dim=0)

    def text(self, literal_idx):
        return ' AND '.join(f"[{c} = '{v}']" for c, v in
                            (self.literals_[i] for i in literal_idx))

# Quick sanity check
enc_full = LiteralEncoder().fit(X_disc)
M_full   = enc_full.encode(X_disc)
print(f'Literal tensor M : {tuple(M_full.shape)}  dtype={M_full.dtype}')
print(f'Total literals   : {len(enc_full.literals_)}')

---
## Stage 3 — Sequential Covering
*Author: Chunarkar Sujeeth Kumar (24AI10021)*

RIPPER-style **separate-and-conquer** loop:
1. **Grow** a rule greedily using FOIL information gain
2. **Prune** trailing conditions using Cohen's metric *(p − n) / (p + n)*
3. **Remove** covered positives and repeat
4. A **default rule** handles the residual

All gain/count operations are tensor reductions — no Python loop over examples.

In [ ]:
class SequentialCovering:
    def __init__(self, max_rules=10, max_conds=2, prune_fraction=0.33,
                 beam_width=1, random_state=42):
        self.max_rules      = max_rules
        self.max_conds      = max_conds
        self.prune_fraction = prune_fraction
        self.beam_width     = beam_width
        self.random_state   = random_state

    # --- grow one rule on a (masked) dataset ---
    def _grow(self, M, y, enc):
        covered = torch.ones(M.shape[1], dtype=torch.bool)
        rule = []
        while len(rule) < self.max_conds:
            p0 = int((covered & y).sum())
            n0 = int((covered & ~y).sum())
            if p0 == 0 or n0 == 0:
                break
            C  = M & covered                          # (L, N) candidates
            p1 = (C & y).sum(1).double()
            n1 = (C & ~y).sum(1).double()
            prec0 = p0 / (p0 + n0 + 1e-9)
            prec1 = p1 / (p1 + n1 + 1e-9)
            gain  = p1 * (torch.log2(prec1.clamp_min(1e-12)) - math.log2(prec0))
            gain[(p1 == 0) | (prec1 <= prec0)] = -1.0
            if rule: gain[rule] = -1.0
            best = int(torch.argmax(gain))
            if float(gain[best]) <= 0:
                break
            rule.append(best)
            covered = C[best]
        return rule

    def _cohen(self, M, y, lits, enc):
        m = enc.rule_mask(M, lits)
        p, n = int((m & y).sum()), int((m & ~y).sum())
        if p + n == 0: return -1.0, p, n
        return (p - n) / (p + n), p, n

    def _prune(self, M, y, rule, enc):
        best_s, best_p, best_n = self._cohen(M, y, rule, enc)
        best_rule = list(rule)
        for k in range(len(rule) - 1, 0, -1):
            s, p, n = self._cohen(M, y, rule[:k], enc)
            if s >= best_s:
                best_s, best_p, best_n, best_rule = s, p, n, rule[:k]
        if best_p <= best_n: return None
        return best_rule

    def fit(self, X_df, y, enc):
        g = torch.Generator().manual_seed(self.random_state)
        M = enc.encode(X_df)
        y = torch.from_numpy(np.asarray(y).astype(bool))
        alive = torch.ones(M.shape[1], dtype=torch.bool)
        self.rules_, self.stats_ = [], []
        while len(self.rules_) < self.max_rules:
            pos = torch.nonzero(alive & y).squeeze(1)
            neg = torch.nonzero(alive & ~y).squeeze(1)
            if len(pos) == 0 or len(neg) == 0: break
            pos = pos[torch.randperm(len(pos), generator=g)]
            neg = neg[torch.randperm(len(neg), generator=g)]
            pp = max(1, int(len(pos) * self.prune_fraction))
            nn = max(1, int(len(neg) * self.prune_fraction))
            prune_idx = torch.cat([pos[:pp], neg[:nn]])
            grow_idx  = torch.cat([pos[pp:], neg[nn:]])
            grown  = self._grow(M[:, grow_idx], y[grow_idx], enc)
            if not grown: break
            pruned = self._prune(M[:, prune_idx], y[prune_idx], grown, enc)
            if not pruned: break
            cov    = enc.rule_mask(M, pruned) & alive
            p_full = int((cov & y).sum())
            n_full = int((cov & ~y).sum())
            if p_full == 0: break
            self.rules_.append(pruned)
            self.stats_.append({'p': p_full, 'n': n_full,
                                'precision': p_full / (p_full + n_full)})
            alive = alive & ~cov
        n_alive = int(alive.sum())
        self.default_prob_ = float((alive & y).sum()) / n_alive if n_alive else 0.0
        self._enc = enc
        return self

    def predict_proba(self, X_df):
        M = self._enc.encode(X_df)
        probs = torch.full((M.shape[1],), self.default_prob_, dtype=torch.double)
        uncovered = torch.ones(M.shape[1], dtype=torch.bool)
        for idx, st in zip(self.rules_, self.stats_):
            hit = self._enc.rule_mask(M, idx) & uncovered
            probs[hit] = st['precision']
            uncovered &= ~hit
        return probs.numpy()

    def print_rules(self, enc):
        print(f'Sequential Covering ({len(self.rules_)} rules):')
        for i, (idx, st) in enumerate(zip(self.rules_, self.stats_), 1):
            print(f'  Rule {i}: IF {enc.text(idx)} -> >50K'
                  f'  (prec={st["precision"]:.3f}, pos={st["p"]}, neg={st["n"]})')
        print(f'  DEFAULT: ELSE <=50K  (residual pos rate={self.default_prob_:.3f})')

### 3.4 Training & Evaluation (80/20 split)

In [ ]:
X_disc_all = discretise(X_raw)
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(
    X_disc_all, y, test_size=0.20, random_state=42, stratify=y)

enc3 = LiteralEncoder().fit(X_tr3)
sc   = SequentialCovering(random_state=42).fit(X_tr3, y_tr3, enc3)
sc.print_rules(enc3)

sc_prob3 = sc.predict_proba(X_te3)
sc_pred3 = (sc_prob3 >= 0.5).astype(int)
print('\nTest metrics:')
print(f'  Accuracy  : {accuracy_score(y_te3, sc_pred3):.4f}')
print(f'  Precision : {precision_score(y_te3, sc_pred3, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_te3, sc_pred3, zero_division=0):.4f}')
print(f'  F1        : {f1_score(y_te3, sc_pred3, zero_division=0):.4f}')
print(f'  AUC       : {roc_auc_score(y_te3, sc_prob3):.4f}')

---
## Stage 4 — Bayesian Rule List
*Author: Mohammad Asad ur Rahman Nisar (24AI10038)*

An ordered decision list learned by maximising a Bayesian posterior:
- **Likelihood**: Dirichlet-Multinomial marginal per leaf (via `lgamma`)
- **Prior**: Truncated-Poisson on list length + width penalty for 2-literal rules
- **Search**: Metropolis-Hastings with 4 symmetric moves: *add, remove, swap, replace*

In [ ]:
class BayesianRuleList:
    def __init__(self, alpha0=1.0, alpha1=1.0, list_prior_lambda=3.0,
                 width_penalty=0.5, max_len=8, minsupport=0.03,
                 n_iter=4000, random_state=42):
        self.alpha0        = alpha0
        self.alpha1        = alpha1
        self.lam           = list_prior_lambda
        self.width_penalty = width_penalty
        self.max_len       = max_len
        self.minsupport    = minsupport
        self.n_iter        = n_iter
        self.random_state  = random_state

    # --- frequent 1- and 2-literal antecedent candidates ---
    def _mine(self, M, enc, min_count):
        counts = M.sum(1)
        sing   = torch.nonzero(counts >= min_count).squeeze(1)
        Ms     = M[sing].float()
        joint  = Ms @ Ms.T
        col    = enc.literal_col_[sing]
        valid  = (joint >= min_count) & (col[:, None] != col[None, :])
        valid  = torch.triu(valid, diagonal=1)
        pi, pj = torch.nonzero(valid, as_tuple=True)
        self.cand_ = [(enc.literals_[int(i)],) for i in sing]
        masks = [M[sing]]
        self.cand_ += [(enc.literals_[int(sing[a])], enc.literals_[int(sing[b])])
                       for a, b in zip(pi.tolist(), pj.tolist())]
        if len(pi):
            masks.append(M[sing[pi]] & M[sing[pj]])
        return torch.cat(masks, 0)

    # --- Dirichlet-Multinomial log marginal likelihood of one leaf ---
    def _log_leaf(self, n0, n1):
        a0, a1 = self.alpha0, self.alpha1
        return (math.lgamma(a0+a1) - math.lgamma(a0) - math.lgamma(a1)
                + math.lgamma(n0+a0) + math.lgamma(n1+a1)
                - math.lgamma(n0+n1+a0+a1))

    # --- log posterior of a list ---
    def _log_posterior(self, order, cm, y):
        covered = torch.zeros(cm.shape[1], dtype=torch.bool)
        loglik, probs, counts = 0.0, [], []
        for idx in order:
            m  = cm[idx] & ~covered
            n1 = int((m & y).sum());  n0 = int((m & ~y).sum())
            counts.append((n0, n1))
            probs.append((n1 + self.alpha1) / (n0+n1+self.alpha0+self.alpha1))
            loglik += self._log_leaf(n0, n1)
            covered |= m
        d  = ~covered
        n1 = int((d & y).sum());  n0 = int((d & ~y).sum())
        counts.append((n0, n1))
        probs.append((n1+self.alpha1) / (n0+n1+self.alpha0+self.alpha1))
        loglik += self._log_leaf(n0, n1)
        m_len     = len(order)
        log_prior = m_len * math.log(self.lam) - math.lgamma(m_len + 1)
        for idx in order:
            log_prior -= self.width_penalty * (len(self.cand_[idx]) - 1)
        return loglik + log_prior, probs, counts

    # --- Metropolis-Hastings search ---
    def _mh(self, cm, y, seed):
        rng  = random.Random(seed)
        n_c  = cm.shape[0]
        best_c, best_s = 0, -1e18
        for i in range(n_c):
            s, _, _ = self._log_posterior([i], cm, y)
            if s > best_s: best_c, best_s = i, s
        cur = [best_c]
        cur_s, cur_p, cur_n = self._log_posterior(cur, cm, y)
        best = (list(cur), cur_s, cur_p, cur_n)
        def other(exclude):
            for _ in range(50):
                c = rng.randrange(n_c)
                if c not in exclude: return c
            return None
        for _ in range(self.n_iter):
            move = rng.randrange(4)
            prop = list(cur)
            if   move == 0 and len(prop) < self.max_len:
                c = other(set(cur))
                if c is None: continue
                prop.insert(rng.randint(0, len(prop)), c)
            elif move == 1 and len(prop) > 1:
                prop.pop(rng.randrange(len(prop)))
            elif move == 2 and len(prop) > 1:
                i, j = rng.sample(range(len(prop)), 2)
                prop[i], prop[j] = prop[j], prop[i]
            elif move == 3:
                c = other(set(cur))
                if c is None: continue
                prop[rng.randrange(len(prop))] = c
            else: continue
            ps, pp, pn = self._log_posterior(prop, cm, y)
            if math.log(rng.random() + 1e-12) < (ps - cur_s):
                cur, cur_s, cur_p, cur_n = prop, ps, pp, pn
                if cur_s > best[1]:
                    best = (list(cur), cur_s, cur_p, cur_n)
        return best

    def fit(self, X_df, y, enc):
        self._enc = enc
        M = enc.encode(X_df)
        y = torch.from_numpy(np.asarray(y).astype(bool))
        self.cand_masks_ = self._mine(M, enc, int(self.minsupport * M.shape[1]))
        best = self._mh(self.cand_masks_, y, self.random_state)
        self.best_order_, self.best_score_, self.leaf_probs_, self.leaf_counts_ = best
        return self

    def predict_proba(self, X_df):
        M = self._enc.encode(X_df)
        probs = torch.full((M.shape[1],), self.leaf_probs_[-1], dtype=torch.double)
        uncovered = torch.ones(M.shape[1], dtype=torch.bool)
        for cidx, p in zip(self.best_order_, self.leaf_probs_[:-1]):
            lits = [self._enc.literals_.index(l) for l in self.cand_[cidx]]
            hit  = self._enc.rule_mask(M, lits) & uncovered
            probs[hit] = p
            uncovered &= ~hit
        return probs.numpy()

    def print_rules(self):
        print(f'BRL ({len(self.best_order_)} rules + DEFAULT):')
        for i, (ci, p, cnt) in enumerate(zip(self.best_order_,
                                              self.leaf_probs_[:-1],
                                              self.leaf_counts_[:-1]), 1):
            cond = ' AND '.join(f"[{c} = '{v}']" for c, v in self.cand_[ci])
            print(f'  Rule {i}: IF {cond} -> P(>50K)={p:.3f}  (n0={cnt[0]}, n1={cnt[1]})')
        print(f'  DEFAULT: P(>50K)={self.leaf_probs_[-1]:.3f}'
              f'  (n0={self.leaf_counts_[-1][0]}, n1={self.leaf_counts_[-1][1]})')

### 4.5 Training & Evaluation (80/20 split)

In [ ]:
# Reuse X_tr3 / X_te3 / y_tr3 / y_te3 / enc3 from Stage 3
print('Training BRL (this takes ~30s)...')
brl = BayesianRuleList(random_state=42).fit(X_tr3, y_tr3, enc3)
brl.print_rules()

brl_prob3 = brl.predict_proba(X_te3)
brl_pred3 = (brl_prob3 >= 0.5).astype(int)
print('\nTest metrics:')
print(f'  Accuracy  : {accuracy_score(y_te3, brl_pred3):.4f}')
print(f'  Precision : {precision_score(y_te3, brl_pred3, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_te3, brl_pred3, zero_division=0):.4f}')
print(f'  F1        : {f1_score(y_te3, brl_pred3, zero_division=0):.4f}')
print(f'  AUC       : {roc_auc_score(y_te3, brl_prob3):.4f}')

---
## Stage 5 — Evaluation & Audit
*Author: Kotikala Yashwant (24AI10034)*

Protocol: **60/20/20** train/validation/test split, repeated over 3 seeds.
- Classification threshold is tuned on **validation only** (best minority F1)
- Test set is scored **once** — never used to pick any parameter

In [ ]:
def compute_metrics(y_true, prob, tau):
    pred = (prob >= tau).astype(int)
    return dict(acc=accuracy_score(y_true, pred),
                prec=precision_score(y_true, pred, zero_division=0),
                rec=recall_score(y_true, pred, zero_division=0),
                f1=f1_score(y_true, pred, zero_division=0),
                auc=roc_auc_score(y_true, prob))

def best_tau(y_val, prob_val):
    grid = np.unique(np.round(prob_val, 4))
    best_t, best_f = 0.5, -1.0
    for t in grid:
        f = f1_score(y_val, (prob_val >= t).astype(int), zero_division=0)
        if f > best_f: best_t, best_f = float(t), f
    return best_t

X_disc_all = discretise(X_raw)
seeds = (42, 43, 44)
rows  = []

for s in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_disc_all, y, test_size=0.20, random_state=s, stratify=y)
    X_sub, X_val, y_sub, y_val = train_test_split(
        X_tr, y_tr, test_size=0.25, random_state=s, stratify=y_tr)
    enc = LiteralEncoder().fit(X_sub)
    sc_s  = SequentialCovering(random_state=s).fit(X_sub, y_sub, enc)
    brl_s = BayesianRuleList(n_iter=4000, random_state=s).fit(X_sub, y_sub, enc)
    for name, model in [('SeqCovering', sc_s), ('BRL', brl_s)]:
        p_val = model.predict_proba(X_val)
        p_te  = model.predict_proba(X_te)
        tau   = best_tau(y_val, p_val)
        if name == 'SeqCovering':
            n_rules = len(model.rules_)
            n_lits  = sum(len(r) for r in model.rules_)
        else:
            n_rules = len(model.best_order_)
            n_lits  = sum(len(model.cand_[i]) for i in model.best_order_)
        fixed = compute_metrics(y_te, p_te, 0.5)
        tuned = compute_metrics(y_te, p_te, tau)
        row   = dict(model=name, seed=s, rules=n_rules, literals=n_lits, tau=round(tau,3))
        row.update({f'fixed_{k}': round(v,4) for k, v in fixed.items()})
        row.update({f'tuned_{k}': round(v,4) for k, v in tuned.items()})
        rows.append(row)
    print(f'  Seed {s} done')

results_df = pd.DataFrame(rows)
print('\n=== Per-run results ===')
print(results_df[['model','seed','rules','literals','tau',
                   'fixed_acc','fixed_f1','fixed_auc','tuned_f1']].to_string(index=False))

In [ ]:
print('\n=== Mean ± Std over seeds ===')
for model_name in ['SeqCovering', 'BRL']:
    sub = results_df[results_df.model == model_name]
    print(f'\n{model_name}:')
    for col in ['rules','literals','fixed_acc','fixed_prec','fixed_rec',
                'fixed_f1','tuned_f1','fixed_auc']:
        print(f'  {col:14s}: {sub[col].mean():.4f} ± {sub[col].std():.4f}')

---
## Stage 6 — Visualisations
*Author: Kotikala Yashwant (24AI10034)*

All figures use the fixed 80/20 split from Stage 3/4 for reproducibility.

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────
def _box(ax, x, y, w, h, text, color):
    patch = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02',
                            linewidth=1.2, edgecolor='#334155', facecolor=color)
    ax.add_patch(patch)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=8.5, wrap=True)

def plot_decision_list(rows, title, ax):
    n = len(rows)
    ax.set_xlim(0, 1); ax.set_ylim(0, n+0.5); ax.axis('off')
    for i, (cond, outcome) in enumerate(rows):
        y_pos = n - i - 0.5
        is_def = i == n-1
        fill = '#f1f5f9' if is_def else ('#dbeafe' if i%2==0 else '#fce7f3')
        label = ('ELSE' if is_def else f'Rule {i+1}') + '\n' + cond
        _box(ax, 0.04, y_pos, 0.68, 0.70, label, fill)
        ax.text(0.75, y_pos+0.35, outcome, ha='left', va='center', fontsize=8.5,
                color='#1e293b')
    ax.set_title(title, fontsize=11, fontweight='bold', color='#1e3a5f')

# Build rows
sc_rows = [(f'IF {enc3.text(idx)}', f'P(>50K)={st["precision"]:.3f}')
           for idx, st in zip(sc.rules_, sc.stats_)]
sc_rows.append((f'residual P(>50K)={sc.default_prob_:.3f}', 'DEFAULT'))

brl_rows = []
for ci, p in zip(brl.best_order_, brl.leaf_probs_[:-1]):
    cond = ' AND '.join(f"{c}={v}" for c,v in brl.cand_[ci])
    brl_rows.append((f'IF {cond}', f'P(>50K)={p:.3f}'))
brl_rows.append((f'P(>50K)={brl.leaf_probs_[-1]:.3f}', 'DEFAULT'))

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
plot_decision_list(sc_rows,  'Sequential Covering Decision List', axes[0])
plot_decision_list(brl_rows, 'Bayesian Rule List Decision List',  axes[1])
fig.suptitle('Learned Decision Lists — IML Group 11', fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
fig.savefig('results/figures/decision_lists.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved results/figures/decision_lists.png')

In [ ]:
# --- NB proba for ROC comparison ---
codes3 = {col: {v: i for i, v in enumerate(pd.unique(X_tr3[col]))} for col in X_tr3.columns}
def enc_nb(frame):
    return np.array([[codes3[c].get(v,0) for c,v in zip(frame.columns, row)]
                     for row in frame.values], dtype=int)
nb3 = CategoricalNB().fit(enc_nb(X_tr3), y_tr3)
nb_prob3 = nb3.predict_proba(enc_nb(X_te3))[:,1]

# --- ROC curves ---
fig, ax = plt.subplots(figsize=(7, 6))
for name, prob, color in [('Sequential Covering', sc_prob3, '#3730a3'),
                            ('Bayesian Rule List',  brl_prob3,'#0d9488'),
                            ('Naive Bayes',         nb_prob3, '#d97706')]:
    fpr, tpr, _ = roc_curve(y_te3, prob)
    auc = roc_auc_score(y_te3, prob)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Chance')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate (>50K)')
ax.set_title('ROC Curves — Held-out Test Set', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig('results/figures/roc_curves.png', dpi=140)
plt.show()

In [ ]:
# --- Confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, prob) in zip(axes, [('Sequential Covering', sc_prob3),
                                    ('Bayesian Rule List',  brl_prob3)]):
    cm = confusion_matrix(y_te3, (prob>=0.5).astype(int))
    ax.imshow(cm, cmap='Blues')
    ax.set_title(f'{name} (τ=0.50)', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_xticks([0,1]); ax.set_xticklabels(['<=50K','>50K'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['<=50K','>50K'])
    for i in range(2):
        for j in range(2):
            v = cm[i,j]
            ax.text(j,i,str(v),ha='center',va='center',fontsize=13,fontweight='bold',
                    color='white' if v > cm.max()/2 else 'black')
fig.suptitle('Confusion Matrices — Held-out Test Set', fontweight='bold')
plt.tight_layout()
fig.savefig('results/figures/confusion_matrices.png', dpi=140)
plt.show()

In [ ]:
# --- Calibration plot ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0,1],[0,1],'k--',lw=1,label='Perfect calibration')
bins = np.linspace(0, 1, 9)
for name, prob, color in [('Sequential Covering', sc_prob3, '#3730a3'),
                            ('Bayesian Rule List',  brl_prob3,'#0d9488')]:
    bin_idx = np.digitize(prob, bins) - 1
    means, obs = [], []
    for b in range(8):
        sel = bin_idx == b
        if sel.sum() >= 10:
            means.append(prob[sel].mean())
            obs.append(y_te3[sel].mean())
    ax.plot(means, obs, 'o-', lw=2, ms=6, color=color, label=name)
ax.set_xlabel('Mean predicted P(>50K)'); ax.set_ylabel('Observed frequency')
ax.set_title('Calibration / Reliability Diagram', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig('results/figures/calibration.png', dpi=140)
plt.show()

---
## Summary

| Model | Rules | Literals | Accuracy | Precision | Recall | F1 (0.5) | F1 (tuned) | AUC |
|---|---|---|---|---|---|---|---|---|
| Sequential Covering | 3.7 | 7.3 | 0.8246 | 0.7006 | 0.5193 | 0.5953 | 0.5726 | 0.7279 |
| Bayesian Rule List  | 8.0 | 14.0 | 0.8276 | 0.7291 | 0.5022 | 0.5891 | 0.6310 | 0.8578 |

**Key takeaways:**
- Both beat the majority-class baseline (accuracy 0.751)
- BRL has substantially better AUC (0.858 vs 0.728) due to finer probability resolution across 8 leaves
- Sequential Covering is more compact (3.7 rules / 7.3 literals) and faster to interpret
- Threshold tuning helped BRL (+0.04 F1) but not SeqCovering (coarse probability scale)

**Limitations:**
- BRL uses 4,000 MH iterations — the best-found list, not the proven posterior mode
- Conclusions are specific to the UCI Adult dataset
- `race` and `sex` are features; learned associations are not causal claims